# Indian Air Quality Intelligence Platform

## Notebook 02 – Data Validation

### Objectives

- Validate the integrity of the raw dataset
- Identify missing values
- Identify duplicate records
- Verify schema and data types
- Analyze date range and coverage
- Generate a validation summary

**Note:** No cleaning or preprocessing is performed in this notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
DATA_PATH = Path("../data/raw/INDIA_AQI_COMPLETE_20251126.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"{DATA_PATH} not found.")

df = pd.read_csv(DATA_PATH)

print("✅ Dataset Loaded Successfully")

✅ Dataset Loaded Successfully


In [3]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(f"Rows               : {df.shape[0]:,}")
print(f"Columns            : {df.shape[1]}")
print(f"Memory Usage (MB)  : {df.memory_usage(deep=True).sum()/(1024**2):.2f}")

DATASET SUMMARY
Rows               : 842,160
Columns            : 71
Memory Usage (MB)  : 856.96


In [4]:
dtype_summary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str)
})

dtype_summary

,Column,Data Type
City,City,object
State,State,object
Latitude,Latitude,float64
Longitude,Longitude,float64
Datetime,Datetime,object
Year,Year,int64
Month,Month,int64
Day,Day,int64
Hour,Hour,int64
Day_of_Week,Day_of_Week,int64


In [5]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2)
})

missing = missing.sort_values(
    by="Missing %",
    ascending=False
)

missing

,Missing Values,Missing %
Temp_180m_C,842160,100.00
Temp_80m_C,842160,100.00
Wind_Speed_120m_kmh,842160,100.00
Wind_Speed_80m_kmh,842160,100.00
NH3_ugm3,842160,100.00
Temp_120m_C,842160,100.00
UV_Index,842160,100.00
Inversion_Strength_C,842160,100.00
AQI_Category,2516,0.30
EU_AQI_PM10,145,0.02


In [14]:
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2)
})

completely_missing = missing_summary[
    missing_summary["Missing %"] == 100
]

partially_missing = missing_summary[
    (missing_summary["Missing %"] > 0) &
    (missing_summary["Missing %"] < 100)
]

print(f"Columns with 100% missing values : {len(completely_missing)}")
print(f"Columns with partial missing values : {len(partially_missing)}")

display(completely_missing)
display(partially_missing)

Columns with 100% missing values : 8
Columns with partial missing values : 8


,Column,Missing Values,Missing %
Temp_80m_C,Temp_80m_C,842160,100.00
Temp_120m_C,Temp_120m_C,842160,100.00
Temp_180m_C,Temp_180m_C,842160,100.00
Wind_Speed_80m_kmh,Wind_Speed_80m_kmh,842160,100.00
Wind_Speed_120m_kmh,Wind_Speed_120m_kmh,842160,100.00
UV_Index,UV_Index,842160,100.00
NH3_ugm3,NH3_ugm3,842160,100.00
Inversion_Strength_C,Inversion_Strength_C,842160,100.00


,Column,Missing Values,Missing %
US_AQI,US_AQI,145,0.02
US_AQI_PM25,US_AQI_PM25,145,0.02
US_AQI_PM10,US_AQI_PM10,145,0.02
US_AQI_O3,US_AQI_O3,73,0.01
EU_AQI,EU_AQI,145,0.02
EU_AQI_PM25,EU_AQI_PM25,145,0.02
EU_AQI_PM10,EU_AQI_PM10,145,0.02
AQI_Category,AQI_Category,2516,0.30


In [6]:
duplicates = df.duplicated().sum()

print(f"Duplicate Rows : {duplicates:,}")

Duplicate Rows : 0


In [7]:
unique = pd.DataFrame({
    "Unique Values": df.nunique()
})

unique

,Unique Values
City,29
State,29
Latitude,29
Longitude,29
Datetime,29040
Year,4
Month,12
Day,31
Hour,24
Day_of_Week,7


In [17]:
pollutant_columns = [
    col for col in df.columns
    if any(x in col.upper() for x in ["PM", "NO2", "SO2", "CO", "O3", "NH3"])
]

invalid_counts = {}

for col in pollutant_columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        invalid_counts[col] = (df[col] < 0).sum()

pd.DataFrame.from_dict(
    invalid_counts,
    orient="index",
    columns=["Negative Values"]
).sort_values("Negative Values", ascending=False)

,Negative Values
O3_ugm3,82
SO2_ugm3,4
NO2_ugm3,2
US_AQI_PM25,0
EU_AQI_PM25,0
US_AQI_CO,0
US_AQI_O3,0
US_AQI_NO2,0
US_AQI_PM10,0
Cloud_Cover_Percent,0


In [15]:
constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) == 1]

print(f"Constant Columns: {len(constant_cols)}")
constant_cols

Constant Columns: 9


['Temp_80m_C',
 'Temp_120m_C',
 'Temp_180m_C',
 'Wind_Speed_80m_kmh',
 'Wind_Speed_120m_kmh',
 'UV_Index',
 'NH3_ugm3',
 'Temp_Inversion',
 'Inversion_Strength_C']

In [8]:
print(f"Cities   : {df['City'].nunique()}")
print(f"States   : {df['State'].nunique()}")

Cities   : 29
States   : 29


In [9]:
df["Datetime"] = pd.to_datetime(df["Datetime"])

print("Start Date :", df["Datetime"].min())
print("End Date   :", df["Datetime"].max())

Start Date : 2022-08-05 00:00:00
End Date   : 2025-11-26 23:00:00


In [10]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

print(f"Total Numerical Features : {len(numerical_cols)}")

numerical_cols

Total Numerical Features : 61


['Latitude',
 'Longitude',
 'Year',
 'Month',
 'Day',
 'Hour',
 'Day_of_Week',
 'Week_of_Year',
 'Is_Weekend',
 'Quarter',
 'Temp_2m_C',
 'Temp_80m_C',
 'Temp_120m_C',
 'Temp_180m_C',
 'Humidity_Percent',
 'Dew_Point_C',
 'Wind_Speed_10m_kmh',
 'Wind_Speed_80m_kmh',
 'Wind_Speed_120m_kmh',
 'Wind_Dir_10m',
 'Wind_Gusts_kmh',
 'Wind_Stagnation',
 'Precipitation_mm',
 'Rain_mm',
 'Is_Raining',
 'Heavy_Rain',
 'Pressure_MSL_hPa',
 'Surface_Pressure_hPa',
 'Solar_Radiation_Wm2',
 'Direct_Radiation_Wm2',
 'Diffuse_Radiation_Wm2',
 'UV_Index',
 'Cloud_Cover_Percent',
 'Cloud_Low_Percent',
 'Cloud_Mid_Percent',
 'Cloud_High_Percent',
 'Is_Daytime',
 'Sunshine_Seconds',
 'PM2_5_ugm3',
 'PM10_ugm3',
 'PM_Ratio',
 'CO_ugm3',
 'NO2_ugm3',
 'SO2_ugm3',
 'O3_ugm3',
 'Dust_ugm3',
 'NH3_ugm3',
 'AOD',
 'US_AQI',
 'US_AQI_PM25',
 'US_AQI_PM10',
 'US_AQI_NO2',
 'US_AQI_O3',
 'US_AQI_CO',
 'EU_AQI',
 'EU_AQI_PM25',
 'EU_AQI_PM10',
 'Temp_Inversion',
 'Inversion_Strength_C',
 'Festival_Period',
 'Crop_Bu

In [11]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()

print(f"Total Categorical Features : {len(categorical_cols)}")

categorical_cols

Total Categorical Features : 9


['City',
 'State',
 'Day_Name',
 'Season',
 'Time_of_Day',
 'Humidity_Category',
 'Wind_Category',
 'AQI_Category',
 'PM25_Category_India']

In [16]:
high_cardinality = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": df.nunique()
})

high_cardinality = high_cardinality.sort_values(
    by="Unique Values",
    ascending=False
)

high_cardinality.head(15)

,Column,Unique Values
PM_Ratio,PM_Ratio,126158
Sunshine_Seconds,Sunshine_Seconds,62326
Datetime,Datetime,29040
CO_ugm3,CO_ugm3,5024
SO2_ugm3,SO2_ugm3,2512
Dust_ugm3,Dust_ugm3,2504
NO2_ugm3,NO2_ugm3,2120
PM10_ugm3,PM10_ugm3,1945
Surface_Pressure_hPa,Surface_Pressure_hPa,1928
US_AQI_PM10,US_AQI_PM10,1527


In [12]:
validation_report = pd.DataFrame({

    "Metric": [

        "Rows",

        "Columns",

        "Duplicate Rows",

        "Cities",

        "States",

        "Start Date",

        "End Date"

    ],

    "Value": [

        df.shape[0],

        df.shape[1],

        duplicates,

        df["City"].nunique(),

        df["State"].nunique(),

        df["Datetime"].min(),

        df["Datetime"].max()

    ]

})

validation_report

,Metric,Value
0,Rows,842160
1,Columns,71
2,Duplicate Rows,0
3,Cities,29
4,States,29
5,Start Date,2022-08-05 00:00:00
6,End Date,2025-11-26 23:00:00


In [13]:
validation_report.to_csv(
    "../reports/tables/data_validation_report.csv",
    index=False
)

print("✅ Validation Report Saved")

✅ Validation Report Saved
